In [2]:
import os
from datetime import datetime
import sys
from pathlib import Path
import shutil
import zipfile
import rarfile
import tarfile
import py7zr
import pandas as pd
#!pip install thefuzz Levenshtein
from thefuzz import process

In [2]:
def dir_content(dir_path):
    subdir_list = []
    dir_content = os.listdir(dir_path)
    for entry in dir_content:
        if entry.is_dir():
            subdir_list.append(entry.name)
    return subdir_list

In [3]:
def move_contents(src_dir, dst_dir, overwrite=True):
    if not os.path.isdir(src_dir):
        raise NotADirectoryError(f"Source must be a directory: {src_dir}")
        
    os.makedirs(dst_dir, exist_ok=True)
    
    for item in src_dir.iterdir():
        src_path = src_dir / item
        dst_path = dst_dir / item
        
        if os.path.isdir(src_path):
            if os.path.isdir(dst_path):
                move_contents(src_path, dst_path)
        else:
            shutil.move(src_path, dst_path)
    
    os.rmdir(src_dir)

In [8]:
def is_base_zipfile(zip_path, filename='BasicFile.csv'):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        namelist = zf.namelist()
        return filename in namelist

In [9]:
def aggregate_basic_zip(curr_path, parent_dir, irrelevant_files_dir, basic_zip_dir, processed_archives_dir, arch_extn):
    # if subdirectories present at current path, move their content to parent directory recursively
    for item in parent_dir.iterdir():
        if item.isdir() and item.name != "00_new_folder":
            move_contents(child_dir, parent_dir)
        
    # now no directories are present at current location
    for item in parent_dir.iterdir():
        # setting path to move the file after processing
        item_name = item.name
        processed_archive = processed_archives_dir / item_name
        basic_zip_dest = basic_zip_dir / item_name
        irrelevant_file = irrelevant_files_dir / item_name
        
        if item.suffix in arch_extn:
            new_folder = curr_path / "00_new_folder"
            os.makedirs(new_folder, exist_ok=True)

            try:
                if is_rar(item):
                    with rarfile.RarFile(item) as rf:
                        rf.extractall(new_folder)
                    shutil.move(item, processed_archive)
                elif is_tar(item):
                    with tarfile.open(item, 'r') as tar:
                        tar.extractall(new_folder)
                    shutil.move(item, processed_archive)
                elif py7zr.is_7zfile(str(item)):
                    with py7zr.SevenZipFile(item, mode='r') as seven_zip:
                        seven_zip.extractall(path=new_folder)
                    shutil.move(item, processed_archive)
                elif is_zip(item):
                    if is_base_zipfile(item):
                        shutil.move(item, basic_zip_dest)
                    else:
                        with zipfile.ZipFile(item, 'r') as zf:
                            zf.extractall(new_folder)
                        shutil.move(item, processed_archive)
                else:
                    shutil.move(item, irrelevant_file)
            except Exception as e:
                shutil.move(item, irrelevant_file)
        else:
            shutil.move(item, irrelevant_file)

    for item in parent_dir.iterdir():
        if item.isdir() and item.name != "00_new_folder":
            aggregate_basic_zip(curr_path, new_folder, irrelevant_files_dir, basic_zip_dir, processed_archives_dir, arch_extn)

In [ ]:
def main():
    curr_path = Path.cwd()
    parent_dir = curr_path / "01_parent_directory"
    irrelevant_files_dir = curr_path / "02_unnecessary_files"
    basic_zip_dir = curr_path / "03_basic_zip_files"
    processed_archives_dir = curr_path / "04_processed_archives"

    arch_extn = ['.zip', '.rar', '.tar', '.gz', '.tgz', '.bz2', '.7z']
    aggregate_basic_zip(curr_path, parent_dir, irrelevant_files_dir, basic_zip_dir, processed_archives_dir, arch_extn)